In [0]:
%pip install lightgbm

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import pandas as pd
import numpy as np

In [0]:
df = spark.table("workspace.default.features").toPandas()
df = df.drop(columns=["calendar_date"])
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 228904 entries, 0 to 228903
Data columns (total 20 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   air_store_id    228904 non-null  object        
 1   visit_date      228904 non-null  datetime64[ns]
 2   visitors        228904 non-null  int32         
 3   day_of_week     228904 non-null  object        
 4   holiday_flg     228904 non-null  int32         
 5   air_genre_name  228904 non-null  object        
 6   air_area_name   228904 non-null  object        
 7   latitude        228904 non-null  float64       
 8   longitude       228904 non-null  float64       
 9   month           228904 non-null  int32         
 10  is_weekend      228904 non-null  int64         
 11  lag_1           228904 non-null  float64       
 12  lag_7           228904 non-null  float64       
 13  lag_14          228904 non-null  float64       
 14  lag_28          228904 non-null  flo

In [0]:
last_date  = df["visit_date"].max()
test_start = last_date - pd.Timedelta(days=35)
is_test = df["visit_date"] >= test_start

train = df[~is_test].copy()
test  = df[is_test].copy()
print("train:", train.shape)
print("test:", test.shape)

train: (203245, 20)
test: (25659, 20)


In [0]:
exclude = ["air_store_id", "visit_date", "visitors", "y"]
feature_cols = df.drop(columns=exclude).columns
print(feature_cols)

Index(['day_of_week', 'holiday_flg', 'air_genre_name', 'air_area_name',
       'latitude', 'longitude', 'month', 'is_weekend', 'lag_1', 'lag_7',
       'lag_14', 'lag_28', 'roll_mean_7', 'roll_std_7', 'roll_mean_28',
       'roll_std_28'],
      dtype='object')


In [0]:
cat_cols = ["day_of_week", "air_genre_name", "air_area_name"]
for col in cat_cols:
    train[col] = train[col].astype("category")
    test[col]  = test[col].astype("category")

In [0]:
import mlflow
mlflow.set_experiment("/Users/s224722274@deakin.edu.au/restaurant-demand-forecasting")

2026/07/02 00:25:06 INFO mlflow.tracking.fluent: Experiment with name '/Users/s224722274@deakin.edu.au/restaurant-demand-forecasting' does not exist. Creating a new experiment.


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/4148368707804204', creation_time=1782951906754, experiment_id='4148368707804204', last_update_time=1782951906754, lifecycle_stage='active', name='/Users/s224722274@deakin.edu.au/restaurant-demand-forecasting', tags={'mlflow.experiment.sourceName': '/Users/s224722274@deakin.edu.au/restaurant-demand-forecasting',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 's224722274@deakin.edu.au',
 'mlflow.ownerId': '74075805519102'}>

In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def score(y_true, y_pred):
    """Returns the three metrics as a dict. y_pred clipped at 0 (no negative visitors)."""
    y_pred = np.clip(y_pred, 0, None)
    return {
        "mae":   mean_absolute_error(y_true, y_pred),
        "rmse":  np.sqrt(mean_squared_error(y_true, y_pred)),
        "rmsle": np.sqrt(mean_squared_error(np.log1p(y_true), np.log1p(y_pred))),
    }

BASELINE_RMSLE = 0.6683 

In [0]:
import lightgbm as lgb

params = {
    "objective": "regression",   
    "n_estimators": 500,         
    "learning_rate": 0.05,       
    "num_leaves": 31,            
    "random_state": 42,          
}

with mlflow.start_run(run_name="lightgbm_default"):
    mlflow.log_params(params)                      

    model = lgb.LGBMRegressor(**params)
    model.fit(train[feature_cols], train["y"], categorical_feature=cat_cols)

    pred_log = model.predict(test[feature_cols])
    pred = np.expm1(pred_log)

    metrics = score(test["visitors"].values, pred)
    mlflow.log_metrics(metrics)                    
    mlflow.lightgbm.log_model(model, name="model")  

    print("LightGBM (default params) on test set:")
    for k, v in metrics.items():
        print(f"  {k.upper():5} = {v:.4f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.014615 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1994
[LightGBM] [Info] Number of data points in the train set: 203245, number of used features: 16
[LightGBM] [Info] Start training from score 2.798794


🔗 View Logged Model at: https://dbc-510a2031-5bd8.cloud.databricks.com/ml/experiments/4148368707804204/models/m-8bc34b03d6f84709bcda7dda1f7b8296?o=7474652526064746
2026/07/02 00:31:40 INFO mlflow.models.model: Model logged without a signature. Signatures are required for Databricks UC model registry as they validate model inputs and denote the expected schema of model outputs. Please set `input_example` parameter when logging the model to auto infer the model signature. To manually set the signature, please visit https://www.mlflow.org/docs/3.8.1/ml/model/signatures.html for instructions on setting signature on models.


LightGBM (default params) on test set:
  MAE   = 7.7078
  RMSE  = 12.7273
  RMSLE = 0.5370


In [0]:
%pip install statsforecast

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 184.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 MB 152.5 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np

df = spark.table("workspace.default.features").toPandas()
df["visit_date"] = pd.to_datetime(df["visit_date"])


sf_df = df[["air_store_id", "visit_date", "visitors"]].rename(
    columns={"air_store_id": "unique_id", "visit_date": "ds", "visitors": "y"}
)

last_date  = sf_df["ds"].max()
test_start = last_date - pd.Timedelta(days=35)

sf_train = sf_df[sf_df["ds"] <  test_start]
sf_test  = sf_df[sf_df["ds"] >= test_start]

MIN_TRAIN_DAYS = 56
counts = sf_train.groupby("unique_id")["y"].count()
ok_stores = counts[counts >= MIN_TRAIN_DAYS].index

sf_train = sf_train[sf_train["unique_id"].isin(ok_stores)]
sf_test  = sf_test[sf_test["unique_id"].isin(ok_stores)]

print("train:", sf_train.shape)
print("test:", sf_test.shape)

train: (202906, 3)
test: (25218, 3)


In [0]:
from statsforecast import StatsForecast
from statsforecast.models import AutoETS

models = [AutoETS(season_length=7)]

sf = StatsForecast(
    models=models,
    freq="D",        
    n_jobs=-1,       
)

forecast = sf.forecast(df=sf_train, h=35)
forecast.head()

,unique_id,ds,AutoETS
0,air_00a91d42b08b08d9,2017-03-18,26.871330
1,air_00a91d42b08b08d9,2017-03-19,36.286366
2,air_00a91d42b08b08d9,2017-03-20,27.669659
3,air_00a91d42b08b08d9,2017-03-21,29.061850
4,air_00a91d42b08b08d9,2017-03-22,26.397518


In [0]:
eval_df = sf_test.merge(forecast, on=["unique_id", "ds"], how="inner")
print("matched rows for scoring:", len(eval_df))

with mlflow.start_run(run_name="autoets"):
    mlflow.log_param("model", "AutoETS(season_length=7)")
    metrics = score(eval_df["y"].values, eval_df["AutoETS"].values)
    mlflow.log_metrics(metrics)

    print("AutoETS on test set:")
    for k, v in metrics.items():
        print(f"  {k.upper():5} = {v:.4f}")

matched rows for scoring: 24397
AutoETS on test set:
  MAE   = 8.7978
  RMSE  = 14.5114
  RMSLE = 0.6225


## Step 5 results: model comparison (test set, final 35 days)

| Model | RMSLE | MAE | RMSE | Coverage |
|-------|-------|-----|------|----------|
| Seasonal-naive baseline | 0.6683 | 9.40 | 14.93 | all stores |
| AutoETS (per-series, weekly) | 0.6225 | 8.80 | 14.51 | stores with ≥8 wks history only |
| **LightGBM (global, features)** | **0.5370** | **7.71** | **12.73** | **all 829 stores** |

**Finding:** LightGBM wins decisively — lower error on every metric *and* it covers every
store, including short-history ones that AutoETS mathematically cannot fit. AutoETS beats the
naive baseline but its per-series design can't exploit cross-store patterns or calendar/lag features, and it's more sensitive to the irregular closed-day gaps.